# Pythia-410m: the regauge test

**GaugeMLA, our method, is the arm `mla_w64_ours_unrope_cost_kl`.** This notebook regauges Pythia-410m, builds MHA2MLA, TransMLA, CARE and GaugeMLA (kit `p410m_rgkv`) and GQA's mean pool (kit `p410m_rgkv_gqa`, the arm `pool`) from each regauged checkpoint, and measures them at conversion: the paper's Figure 1 and Tables 4 and 5 (see `experiments/regauge_test/README.md`).

Step 1 on a **High-RAM CPU** runtime; then switch to an **A100 GPU** runtime (Runtime → Change runtime type), rerun Setup, and run step 2. In step 1 the MLA methods take hours (their fits); GQA's mean pool fits nothing and takes minutes per condition.

Writes to `MyDrive/gaugemla/regauge_test/p410m_rgkv/` and `MyDrive/gaugemla/regauge_test/p410m_rgkv_gqa/`: needs **at least 95 GB free on Google Drive** (the 51 checkpoints take about 85 GB).

Requires a Hugging Face token: add it as the Colab secret `HF_TOKEN` and give this notebook access.

In [ ]:
# @title Setup
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
# The repository is public. While it is private, uncomment these lines and add the Colab secret READ_MLA_TOKEN
# (a GitHub token that can read it):
# from google.colab import userdata
# os.environ["READ_MLA_TOKEN"] = userdata.get("READ_MLA_TOKEN")
# with open("/content/askpass.sh", "w") as f:
#     f.write('#!/bin/sh\necho "$READ_MLA_TOKEN"\n')
# os.chmod("/content/askpass.sh", 0o700)
# os.environ["GIT_ASKPASS"] = "/content/askpass.sh"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if os.path.isdir("/content/repo/.git"):
    subprocess.run(["git", "-C", "/content/repo", "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/anon5159753/gaugemla.git", "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", "src", "colab"], check=True)
sys.path.insert(0, "/content/repo/colab")
import launch

## Which condition and arm is which in the paper

Each `REGAUGE` condition is a column of the paper's regauge tables, and each arm a row, built at each KV cache budget under each condition.

| `REGAUGE` | in the paper | kits |
|---|---|---|
| `identity` | Identity, the unregauged checkpoint | `p410m_rgkv`, `p410m_rgkv_gqa` |
| `kvs` | Per-head scalar (the column Scalar) | `p410m_rgkv`, `p410m_rgkv_gqa` |
| `kvd` | Diagonal (the column Diag.) | `p410m_rgkv`, `p410m_rgkv_gqa` |
| `kvs_xf` | Per-head scalar, the row GaugeMLA, $F_i$ carried over | `p410m_rgkv` (GaugeMLA only) |
| `kvd_xf` | Diagonal, the row GaugeMLA, $F_i$ carried over | `p410m_rgkv` (GaugeMLA only) |

| arm | in the paper | kit | KV cache budgets |
|---|---|---|---|
| `mla_w64_transmla_sigma` | TransMLA | `p410m_rgkv` | 25%, 12.5%, 6.25% |
| `mla_w64_mha2mla_s2norm` | MHA2MLA | `p410m_rgkv` | 25%, 12.5%, 6.25% |
| `mla_w64_care_sigma` | CARE | `p410m_rgkv` | 25%, 12.5%, 6.25% |
| `mla_w64_ours_unrope_cost_kl` | GaugeMLA (our method) | `p410m_rgkv` | 25%, 12.5%, 6.25% |
| `pool` | GQA | `p410m_rgkv_gqa` | 25%, 12.5%, 6.25% |

In [ ]:
# @title 1. Build the initialisations, identity first (High-RAM CPU runtime)
for c in ("identity", "kvs", "kvd", "kvs_xf", "kvd_xf"):
    launch.run("p410m_rgkv", "init", DRY_RUN=False, REGAUGE=c)
# GQA's mean pool
for c in ("identity", "kvs", "kvd"):
    launch.run("p410m_rgkv_gqa", "init", DRY_RUN=False, REGAUGE=c)

In [ ]:
# @title 2. Score them (A100 GPU runtime: switch, rerun Setup, then this cell)
for c in ("identity", "kvs", "kvd", "kvs_xf", "kvd_xf"):
    launch.run("p410m_rgkv", "run", REGAUGE=c)
# GQA's mean pool
for c in ("identity", "kvs", "kvd"):
    launch.run("p410m_rgkv_gqa", "run", REGAUGE=c)

In [ ]:
# @title 3. The numbers, laid out as the paper's Tables 4 and 5, beside the paper's (after step 2; reads Drive)
launch.regauge_table()